In [ ]:
import torch
import numpy as np
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [4]:
data=torch.load("./train_data_with_context/all_conversations.pt")
data2=torch.load("./train_data_with_context/all_conversations2.pt")

In [5]:
data[0][0].keys()

dict_keys(['x_t', 'zt', 'score', 'y'])

In [6]:
X = []
y=[]
for convo in data:
    for turn in convo:
        X.append(torch.concat([turn["x_t"], turn["zt"]], dim=0))
        y.append(turn["score"])

In [7]:
X2 = []
y2=[]
for convo in data2:
    for turn in convo:
        X2.append(torch.concat([turn["x_t"], turn["ut"]], dim=0))
        y2.append(turn["score"])

In [8]:
X = np.array(X)
y = np.array(y)
X2 = np.array(X2)
y2 = np.array(y2)

In [9]:
X_train1, X_test1, y_train1, y_test1 = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X2, y2, test_size=0.2, random_state=42
)

In [ ]:
# Create SVM model
model = SVC(kernel='rbf', C=1.0, gamma='scale')  # default good starting point

# Train
model.fit(X_train1, y_train1)

# Predict
y_pred = model.predict(X_test1)

# Evaluate
print("Accuracy:", accuracy_score(y_test1, y_pred))
print(classification_report(y_test1, y_pred))

Accuracy: 0.5798826777087647
              precision    recall  f1-score   support

           1       0.68      0.69      0.69      1467
           2       0.50      0.30      0.38       927
           3       0.54      0.75      0.63      2173
           4       0.57      0.38      0.45       682
           5       0.64      0.34      0.45       547

    accuracy                           0.58      5796
   macro avg       0.59      0.49      0.52      5796
weighted avg       0.58      0.58      0.56      5796



In [ ]:
from sklearn.model_selection import GridSearchCV

params = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']
}

grid = GridSearchCV(SVC(), params, cv=5)
grid.fit(X_train1, y_train1)

print("Best params:", grid.best_params_)

Best params: {'C': 1, 'gamma': 'scale', 'kernel': 'rbf'}


# with dimensionality reduction

In [42]:
from sklearn.decomposition import PCA

In [ ]:
pca = PCA(n_components=0.95)  # keep 95% of variance
X_train_pca = pca.fit_transform(X_train1)
print("Number of components:", pca.n_components_)
print("Total explained variance:", pca.explained_variance_ratio_.sum())

# Create SVM model
model = SVC(kernel='rbf', C=1.0, gamma='scale')  

# Train
model.fit(X_train_pca, y_train1)

# Predict
y_pred = model.predict(pca.transform(X_test1))

# Evaluate
print("Accuracy:", accuracy_score(y_test1, y_pred))
print(classification_report(y_test1, y_pred))

Number of components: 212
Total explained variance: 0.95000315
Accuracy: 0.5778122843340234
              precision    recall  f1-score   support

           1       0.68      0.69      0.68      1467
           2       0.50      0.30      0.37       927
           3       0.54      0.75      0.63      2173
           4       0.56      0.37      0.45       682
           5       0.63      0.35      0.45       547

    accuracy                           0.58      5796
   macro avg       0.58      0.49      0.52      5796
weighted avg       0.58      0.58      0.56      5796



# trial 2

In [ ]:
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X2, y2, test_size=0.1, random_state=42
)

# Create SVM model
model = SVC(kernel='rbf', C=1.0, gamma='scale')  # default good starting point

# Train
model.fit(X_train2, y_train2)

# Predict
y_pred = model.predict(X_test2)

# Evaluate
print("Accuracy:", accuracy_score(y_test2, y_pred))
print(classification_report(y_test2, y_pred))

Accuracy: 0.5707384403036577
              precision    recall  f1-score   support

           1       0.68      0.67      0.68       718
           2       0.51      0.29      0.37       468
           3       0.53      0.76      0.62      1084
           4       0.55      0.33      0.41       359
           5       0.60      0.36      0.45       269

    accuracy                           0.57      2898
   macro avg       0.57      0.48      0.51      2898
weighted avg       0.57      0.57      0.55      2898



In [ ]:
pca = PCA(n_components=0.95)  # keep 95% of variance
X_train_pca = pca.fit_transform(X_train2)
print("Number of components:", pca.n_components_)
print("Total explained variance:", pca.explained_variance_ratio_.sum())

# Create SVM model
model = SVC(kernel='rbf', C=1.0, gamma='scale')  

# Train
model.fit(X_train_pca, y_train2)

# Predict
y_pred = model.predict(pca.transform(X_test2))

# Evaluate
print("Accuracy:", accuracy_score(y_test2, y_pred))
print(classification_report(y_test2, y_pred))

Number of components: 352
Total explained variance: 0.9501066
Accuracy: 0.5871290545203589
              precision    recall  f1-score   support

           1       0.69      0.68      0.69      1467
           2       0.54      0.33      0.41       927
           3       0.55      0.76      0.64      2173
           4       0.56      0.37      0.45       682
           5       0.62      0.37      0.46       547

    accuracy                           0.59      5796
   macro avg       0.59      0.50      0.53      5796
weighted avg       0.59      0.59      0.57      5796



# logistic

In [ ]:
lr_model = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000)
lr_model.fit(X_train1, y_train1)
lr_preds = lr_model.predict(X_test1)

print("Logistic Regression Accuracy:", accuracy_score(y_test1, lr_preds))
print(classification_report(y_test1, lr_preds))

d:\anaconda\envs\genai-env\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Logistic Regression Accuracy: 0.5567632850241546
              precision    recall  f1-score   support

           1       0.64      0.67      0.65      1467
           2       0.49      0.28      0.36       927
           3       0.53      0.71      0.61      2173
           4       0.52      0.37      0.43       682
           5       0.60      0.36      0.45       547

    accuracy                           0.56      5796
   macro avg       0.55      0.48      0.50      5796
weighted avg       0.55      0.56      0.54      5796



In [ ]:
pca = PCA(n_components=0.95)  
X_train_pca = pca.fit_transform(X_train1)
print("Number of components:", pca.n_components_)
print("Total explained variance:", pca.explained_variance_ratio_.sum())

lr_model = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000)
lr_model.fit(X_train_pca, y_train1)
lr_preds = lr_model.predict(pca.transform(X_test1))

# Evaluate
print("Accuracy:", accuracy_score(y_test1, lr_preds))
print(classification_report(y_test1, lr_preds))

Number of components: 212
Total explained variance: 0.950003150448835


d:\anaconda\envs\genai-env\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Accuracy: 0.5370945479641132
              precision    recall  f1-score   support

           1       0.62      0.64      0.63      1467
           2       0.46      0.27      0.34       927
           3       0.51      0.70      0.59      2173
           4       0.50      0.34      0.41       682
           5       0.57      0.33      0.41       547

    accuracy                           0.54      5796
   macro avg       0.53      0.45      0.48      5796
weighted avg       0.54      0.54      0.52      5796



# second approach

In [ ]:
lr_model = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000)
lr_model.fit(X_train2, y_train2)
lr_preds = lr_model.predict(X_test2)

print("Logistic Regression Accuracy:", accuracy_score(y_test2, lr_preds))
print(classification_report(y_test2, lr_preds))

d:\anaconda\envs\genai-env\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Logistic Regression Accuracy: 0.549344375431332
              precision    recall  f1-score   support

           1       0.64      0.66      0.65      1467
           2       0.48      0.29      0.36       927
           3       0.52      0.70      0.60      2173
           4       0.51      0.35      0.41       682
           5       0.56      0.35      0.43       547

    accuracy                           0.55      5796
   macro avg       0.54      0.47      0.49      5796
weighted avg       0.55      0.55      0.54      5796



In [ ]:
pca = PCA(n_components=0.95)  
X_train_pca = pca.fit_transform(X_train2)
print("Number of components:", pca.n_components_)
print("Total explained variance:", pca.explained_variance_ratio_.sum())

lr_model = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000)
lr_model.fit(X_train_pca, y_train2)
lr_preds = lr_model.predict(pca.transform(X_test2))

# Evaluate
print("Accuracy:", accuracy_score(y_test2, lr_preds))
print(classification_report(y_test2, lr_preds))

Number of components: 352
Total explained variance: 0.9501065192747147


d:\anaconda\envs\genai-env\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Accuracy: 0.5410628019323671
              precision    recall  f1-score   support

           1       0.62      0.64      0.63      1467
           2       0.48      0.28      0.35       927
           3       0.51      0.70      0.59      2173
           4       0.52      0.34      0.41       682
           5       0.56      0.35      0.43       547

    accuracy                           0.54      5796
   macro avg       0.54      0.46      0.48      5796
weighted avg       0.54      0.54      0.53      5796



# xgb

In [67]:
xgb_model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    objective='multi:softmax',
    num_class=5, 
    random_state=42
)

xgb_model.fit(X_train1, y_train1 - 1)
xgb_preds = xgb_model.predict(X_test1) + 1

print("XGBoost Accuracy:", accuracy_score(y_test1, xgb_preds))
print(classification_report(y_test1, xgb_preds))

XGBoost Accuracy: 0.554520358868185
              precision    recall  f1-score   support

           1       0.65      0.66      0.65      1467
           2       0.50      0.27      0.35       927
           3       0.52      0.72      0.60      2173
           4       0.54      0.37      0.44       682
           5       0.59      0.31      0.41       547

    accuracy                           0.55      5796
   macro avg       0.56      0.47      0.49      5796
weighted avg       0.56      0.55      0.54      5796

